# SPEC-01 to SPEC-03: Data Preparation, EDA, and Leakage-Safe Splitting

This notebook is an executable companion to the production code. It demonstrates configured ingestion, validation, immutable source fingerprinting, column-role profiling, explicit target/feature separation, versioned privacy-safe grouped splitting, train-only preprocessing, and deterministic exploratory analysis.

The notebook calls project modules instead of duplicating their implementation. EDA is descriptive and does not establish causation, modify the raw CSV, select model features from holdout performance, or fit a predictive model.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from machine_learning_project.data.ingestion import load_csv, sha256_file
from machine_learning_project.data.preparation import prepare_supervised_data
from machine_learning_project.data.profiling import build_profile
from machine_learning_project.data.splitting import split_by_group, write_split_artifacts
from machine_learning_project.data.validation import require_valid_schema
from machine_learning_project.features.preprocessing import build_preprocessor_from_config
from machine_learning_project.utils.config import (
    load_yaml,
    validate_data_config,
    validate_eda_config,
    validate_preprocessing_config,
)
from pipelines.data_pipeline import run_eda

## 1. Load and validate configuration

The YAML files are the authoritative contracts. Unexpected source columns are rejected, and preprocessing behavior is validated before fitting.

In [ ]:
data_config = load_yaml(PROJECT_ROOT / 'configs/data.yaml')['data']
preprocessing_config = load_yaml(PROJECT_ROOT / 'configs/preprocessing.yaml')['preprocessing']
training_config = load_yaml(PROJECT_ROOT / 'configs/training.yaml')['training']
eda_config = load_yaml(PROJECT_ROOT / 'configs/eda.yaml')['eda']

# Resolve paths so the notebook works from either the project root or notebooks/.
data_config = {**data_config, 'csv_path': str(PROJECT_ROOT / data_config['csv_path'])}
eda_config = {
    **eda_config,
    'output_directory': str(PROJECT_ROOT / eda_config['output_directory']),
    'figures_directory': str(PROJECT_ROOT / eda_config['figures_directory']),
}

validate_data_config(data_config)
validate_preprocessing_config(preprocessing_config)
validate_eda_config(eda_config)
print('Configuration contracts are valid.')

## 2. Ingest, fingerprint, and validate the raw dataset

The SHA-256 value represents the original file bytes. The source is loaded read-only and validated before analysis.

In [ ]:
loaded = load_csv(data_config)
require_valid_schema(loaded.dataframe, data_config)
source_hash_before = loaded.sha256

print(f'Rows: {len(loaded.dataframe):,}')
print(f'Columns: {len(loaded.dataframe.columns)}')
print(f'Source SHA-256: {source_hash_before}')
loaded.dataframe.head()

## 3. Structural profile and column disposition

Every column is assigned an explicit role: target, identifier, approved feature, or dropped field. Sensitive annotations are retained separately from the primary role.

In [ ]:
profile = build_profile(
    loaded.dataframe,
    loaded.sha256,
    data_config,
    feature_contract_version=preprocessing_config['feature_contract_version'],
)

profile_columns = pd.DataFrame.from_dict(profile['columns'], orient='index')
display(profile['column_disposition_counts'])
display(profile_columns[['role', 'dtype', 'null_percentage', 'unique_count', 'model_eligible', 'warnings']])

## 4. Explicit target and feature separation

Identifiers, the target, and all dropped/leakage fields are excluded from the feature matrix by the central contract. Preparation returns defensive copies and preserves row order.

In [ ]:
prepared = prepare_supervised_data(
    loaded.dataframe,
    data_config,
    feature_contract_version=preprocessing_config['feature_contract_version'],
)

assert data_config['target_column'] not in prepared.features.columns
assert not set(data_config['id_columns']) & set(prepared.features.columns)
assert not set(data_config['drop_columns']) & set(prepared.features.columns)
assert prepared.features.index.equals(loaded.dataframe.index)

print(f'Approved raw features: {prepared.features.shape[1]}')
display(pd.Series(prepared.disposition).value_counts().rename('column_count'))

## 5. Grouped split and train-only preprocessing

Splitting happens before learned transformations. Median values, categorical imputations, one-hot vocabularies, and scaling parameters are learned only from the training partition. Validation and test partitions call `transform` only.

In [ ]:
splits = split_by_group(
    loaded.dataframe,
    target_column=data_config['target_column'],
    group_column=training_config['group_column'],
    row_key=training_config['row_key'],
    allowed_labels=data_config['allowed_target_values'],
    test_size=training_config['test_size'],
    validation_size=training_config['validation_size'],
    random_seed=training_config['random_seed'],
    search_attempts=training_config['search_attempts'],
    row_ratio_tolerance=training_config['row_ratio_tolerance'],
    class_ratio_tolerance=training_config['class_ratio_tolerance'],
    size_weight=training_config['split_objective_weights']['size'],
    class_weight=training_config['split_objective_weights']['class'],
    source_sha256=loaded.sha256,
    data_schema_version=data_config['schema_version'],
    split_contract_version=training_config['split_contract_version'],
    algorithm_version=training_config['split_algorithm_version'],
    alias_context=training_config['split_alias_context'],
)

prepared_splits = {
    name: prepare_supervised_data(
        getattr(splits, name),
        data_config,
        feature_contract_version=preprocessing_config['feature_contract_version'],
    )
    for name in ('train', 'validation', 'test')
}

preprocessor = build_preprocessor_from_config(
    data_config['numeric_columns'],
    data_config['categorical_columns'],
    preprocessing_config,
)
transformed = {
    'train': preprocessor.fit_transform(prepared_splits['train'].features),
    'validation': preprocessor.transform(prepared_splits['validation'].features),
    'test': preprocessor.transform(prepared_splits['test'].features),
}

split_summary = pd.DataFrame(
    {
        name: {
            'rows': matrix.shape[0],
            'transformed_features': matrix.shape[1],
            'clients': getattr(splits, name)[training_config['group_column']].nunique(),
        }
        for name, matrix in transformed.items()
    }
).T
display(split_summary)
display(pd.DataFrame(splits.manifest['partitions']).T)
assert splits.manifest['invariants']['row_complete']
assert splits.manifest['invariants']['group_disjoint']
assert len(splits.assignments) == len(loaded.dataframe)
split_manifest_path, split_assignments_path = write_split_artifacts(
    splits,
    PROJECT_ROOT / training_config['split_manifest_path'],
    PROJECT_ROOT / training_config['split_assignments_path'],
)
print(f'Split manifest: {split_manifest_path}')
print(f'Split assignments: {split_assignments_path}')
display(pd.Series(preprocessor.get_feature_names_out(), name='transformed_feature').head(20))

## 6. Run deterministic EDA

This calls the same production pipeline as `scripts/run_eda.py`. It creates versioned tables in `reports/eda/` and aggregate figures in `reports/figures/eda/`. Client values are masked in cohort output and excluded from categorical summaries.

In [ ]:
eda_results, artifact_paths = run_eda(data_config, eda_config)
print(f'Generated {len(artifact_paths)} EDA artifacts.')
display(pd.Series(eda_results.summary, name='value'))
display(pd.DataFrame({'artifact': [str(path) for path in artifact_paths]}))

## 7. Review key findings

Rate-domain checks are provisional until stakeholders confirm whether each field is stored as a fraction, percentage, or another scale. Their violations are warnings, not confirmed source errors.

In [ ]:
target_summary = pd.DataFrame(
    {
        'count': eda_results.summary['target']['counts'],
        'proportion': eda_results.summary['target']['proportions'],
    }
)
display(target_summary)

display(
    eda_results.missingness_summary
    .query("kind == 'column' and count > 0")
    .sort_values('percentage', ascending=False)
    .head(15)
)
display(eda_results.anomalies.query('violation_count > 0'))
display(
    eda_results.associations
    .query("association_type == 'numeric_pair' and high_association")
    .sort_values('spearman', key=lambda values: values.abs(), ascending=False)
    .head(20)
)
display(eda_results.leakage_register.query('model_eligible == False'))

## 8. Immutability and privacy checks

The final checks confirm that the raw source fingerprint is unchanged, every column appears exactly once in the leakage register, and raw client IDs are absent from text-based EDA artifacts.

In [ ]:
assert sha256_file(loaded.source_path) == source_hash_before
assert len(eda_results.leakage_register) == len(loaded.dataframe.columns)
assert eda_results.leakage_register['column'].is_unique

client_values = set(loaded.dataframe[data_config['group_column']].astype(str).unique())
text_artifacts = [path for path in artifact_paths if Path(path).suffix in {'.csv', '.json', '.md'}]
text_artifacts += [split_manifest_path, split_assignments_path]
privacy_hits = []
for artifact in text_artifacts:
    text = Path(artifact).read_text(encoding='utf-8')
    privacy_hits.extend((str(artifact), value) for value in client_values if value in text)

assert not privacy_hits, privacy_hits
print('Passed: source immutability, complete leakage register, and client privacy checks.')

## Current conclusions

- The target is materially imbalanced; macro-averaged model metrics remain necessary.
- High missingness in provider and length metadata requires explicit treatment and monitoring.
- Strong redundancy exists among several traffic, session, and tier representations.
- Outcome-window and target-derived columns remain prohibited model inputs.
- Rate domains, target thresholds, tier boundaries, and aggregation windows still require stakeholder approval.
- Prospective claims require timestamped feature snapshots and future-window labels.